In [2]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import KFold


def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

In [6]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
print(f"Исходные данные загружены. Train: {train.shape}, Test: {test.shape}")


Исходные данные загружены. Train: (751, 214), Test: (250, 211)


In [7]:
target_cols = ['IC50, mM', 'CC50, mM', 'SI']
feature_cols = [col for col in train.columns if col not in ["index"] + target_cols]


In [8]:
medians = train.groupby(feature_cols)[target_cols].transform('median')
train[target_cols] = medians
train_cleaned = train.drop_duplicates(subset=feature_cols, keep='first').reset_index(drop=True)
print(f"После объединения дубликатов молекул по медиане: {train_cleaned.shape}")


После объединения дубликатов молекул по медиане: (630, 214)


In [9]:
train_cleaned = train_cleaned.dropna(subset=target_cols).reset_index(drop=True)
print(f"После удаления строк с пустыми таргетами (NaN): {train_cleaned.shape}")


После удаления строк с пустыми таргетами (NaN): (628, 214)


In [10]:
selector = VarianceThreshold(threshold=0.0)
selector.fit(train_cleaned[feature_cols])
constant_features = [col for col, keep in zip(feature_cols, selector.get_support()) if not keep]
feature_cols = [col for col in feature_cols if col not in constant_features]
print(f"Удалено константных признаков: {len(constant_features)}. Осталось признаков: {len(feature_cols)}")


Удалено константных признаков: 18. Осталось признаков: 192


In [11]:
corr_matrix = train_cleaned[feature_cols].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_features = [column for column in upper_tri.columns if any(upper_tri[column] > 0.95)]
feature_cols = [col for col in feature_cols if col not in high_corr_features]
print(f"Удалено сильно коррелирующих признаков (>0.95): {len(high_corr_features)}. Итого признаков: {len(feature_cols)}")


Удалено сильно коррелирующих признаков (>0.95): 34. Итого признаков: 158


In [12]:
q25 = train_cleaned['SI'].quantile(0.25)
q75 = train_cleaned['SI'].quantile(0.75)
iqr = q75 - q25
upper_boundary = q75 + 3.0 * iqr
train_final = train_cleaned[train_cleaned['SI'] <= upper_boundary].reset_index(drop=True)
print(f"Граница выбросов по IQR для SI: {upper_boundary:.2f}. Удалено выбросов: {train_cleaned.shape[0] - train_final.shape[0]}")


Граница выбросов по IQR для SI: 48.59. Удалено выбросов: 49


In [15]:
train_final['fold'] = -1
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold_idx, (train_idx, val_idx) in enumerate(kf.split(train_final)):
    train_final.loc[val_idx, 'fold'] = fold_idx

print(f"\n Очищенный датасет")
print(f"Размерность train_final: {train_final.shape}")
print(f"Количество признаков в feature_cols: {len(feature_cols)}")
print(f"Распределение строк по 5 фолдам:\n{train_final['fold'].value_counts().to_string()}")


 Очищенный датасет
Размерность train_final: (579, 215)
Количество признаков в feature_cols: 158
Распределение строк по 5 фолдам:
fold
1    116
0    116
2    116
3    116
4    115


C:\Users\Professional\AppData\Local\Temp\ipykernel_17752\3667698995.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_final['fold'] = -1
